In [1]:
"""
Compact BISoN Pipeline for SRKW - Eigenvector Centrality Analysis
Stage 1: Bayesian edge estimation | Stage 2: Propagate to centrality & regression
"""
import warnings; warnings.filterwarnings("ignore")
import pandas as pd
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
from pathlib import Path
from scipy.stats import gaussian_kde
import pymc as pm
import arviz as az
import statsmodels.formula.api as smf

# ============ CONFIG ============
ATTR_PATH = r"C:\Users\srava\Desktop\Random\Codes\Python\Sem 5\code hr\srkw_attributes.csv" 
CONTACT_PATH = r"C:\Users\srava\Desktop\Random\Codes\Python\Sem 5\code hr\srkw_contacts_edgelist.csv"
OUTDIR = Path("srkw_bison_output")
OUTDIR.mkdir(exist_ok=True)
N_DRAWS, N_TUNE, N_CHAINS, N_PROP = 1000, 800, 2, 200
# ================================

def find_col(df, cands):
    return next((c for c in cands if c in df.columns), None)

def load_data():
    attrs, contacts = pd.read_csv(ATTR_PATH), pd.read_csv(CONTACT_PATH)
    contacts = contacts.rename(columns={
        find_col(contacts, ["from", "a", "id1"]): "from",
        find_col(contacts, ["to", "b", "id2"]): "to",
        find_col(contacts, ["count", "contacts", "n"]): "count"
    })
    attrs = attrs.rename(columns={
        find_col(attrs, ["id", "ID", "animal_id"]) or attrs.columns[0]: "id",
        find_col(attrs, ["age", "Age"]): "age",
        find_col(attrs, ["sex", "Sex"]): "sex"
    })
    if "age" in attrs.columns: attrs["age"] = pd.to_numeric(attrs["age"], errors="coerce")
    if "sex" in attrs.columns: attrs["sex"] = attrs["sex"].astype(str).str.upper().str[0]
    print(f"Loaded {len(attrs)} nodes, {len(contacts)} contacts")
    return attrs, contacts

def aggregate_dyads(contacts):
    c = contacts.copy()
    c["u"], c["v"] = c[["from", "to"]].min(axis=1), c[["from", "to"]].max(axis=1)
    return c.groupby(["u", "v"], as_index=False)["count"].sum().rename(columns={"u": "node1", "v": "node2"})

def fit_model(dyads, nodes):
    id2idx = {nid: i for i, nid in enumerate(sorted(nodes))}
    i = dyads["node1"].map(id2idx).astype(int).to_numpy()
    j = dyads["node2"].map(id2idx).astype(int).to_numpy()
    counts = dyads["count"].astype(int).to_numpy()
    
    print(f"\nFitting Bayesian model: {N_DRAWS} draws, {N_TUNE} tune, {N_CHAINS} chains")
    with pm.Model():
        alpha_0 = pm.Normal("alpha_0", mu=np.log(counts.mean()), sigma=5)
        sigma_u = pm.HalfNormal("sigma_u", sigma=2.0)
        u = pm.Normal("u", mu=0, sigma=sigma_u, shape=len(id2idx))
        alpha_nb = pm.Gamma("alpha_nb", mu=2, sigma=1)
        mu = pm.math.exp(alpha_0 + u[i] + u[j])
        pm.NegativeBinomial("obs", mu=mu, alpha=alpha_nb, observed=counts)
        idata = pm.sample(N_DRAWS, tune=N_TUNE, chains=N_CHAINS, target_accept=0.95, 
                         return_inferencedata=True, random_seed=42)
    
    print("Sampling complete!")
    print("\nModel summary:")
    print(az.summary(idata, var_names=["alpha_0", "sigma_u", "alpha_nb"]))
    
    # Extract lambda samples
    post = idata.posterior
    a0 = post["alpha_0"].stack(samples=("chain", "draw")).values
    u_samp = post["u"].stack(samples=("chain", "draw")).values
    lam = np.array([np.exp(a0[s] + u_samp[i, s] + u_samp[j, s]) for s in range(len(a0))])
    return lam, id2idx

def compute_eigen(lam_draw, dyads, id2idx):
    G = nx.Graph()
    for idx, (n1, n2) in enumerate(zip(dyads["node1"], dyads["node2"])):
        G.add_edge(n1, n2, weight=lam_draw[idx])
    for nid in id2idx: 
        if nid not in G: G.add_node(nid)
    try:
        eigen = nx.eigenvector_centrality(G, weight="weight", max_iter=1000)
        return np.array([eigen.get(n, 0) for n in id2idx.keys()])
    except:
        return np.zeros(len(id2idx))

def propagate(lam_samples, dyads, id2idx):
    print(f"\nPropagating {N_PROP} draws to eigenvector centrality...")
    idx_draws = np.random.choice(len(lam_samples), min(N_PROP, len(lam_samples)), replace=False)
    eigen = np.vstack([compute_eigen(lam_samples[i], dyads.reset_index(drop=True), id2idx) 
                       for i in idx_draws])
    print(f"Complete! Shape: {eigen.shape}")
    return eigen

def run_regression(eigen, attrs, nodes):
    print("\nRunning regressions...")
    coefs = []
    for d in range(len(eigen)):
        df = pd.DataFrame({"id": nodes, "eigenvector": eigen[d]}).merge(attrs, on="id", how="left")
        if "sex" in df.columns:
            df["sex_M"] = df["sex"].map(lambda s: 1 if str(s).upper().startswith("M") else 0)
            formula = "eigenvector ~ age + sex_M"
        else:
            formula = "eigenvector ~ age"
        try:
            coefs.append(smf.ols(formula, data=df).fit().params)
        except:
            coefs.append(pd.Series({"Intercept": np.nan, "age": np.nan, "sex_M": np.nan}))
    return pd.DataFrame(coefs)

def save_and_plot(eigen, coefs_df, attrs, nodes, dyads, lam_samples, id2idx):
    # Summary
    summary = pd.DataFrame({col: {"mean": coefs_df[col].mean(), "2.5%": np.nanpercentile(coefs_df[col], 2.5),
                                   "97.5%": np.nanpercentile(coefs_df[col], 97.5)} 
                           for col in coefs_df.columns}).T
    summary.to_csv(OUTDIR / "regression_coefficients.csv")
    
    # Node summary
    node_df = pd.DataFrame({"id": nodes, "eigenvector_mean": eigen.mean(axis=0)})
    node_df = node_df.merge(attrs, on="id", how="left")
    node_df.to_csv(OUTDIR / "node_eigenvector_summary.csv", index=False)
    
    print("\n" + "="*60)
    print("REGRESSION RESULTS: Eigenvector Centrality ~ Age + Sex")
    print("="*60)
    print(summary)
    print("="*60)
    print(f"\nTop 5 nodes:")
    print(node_df.nlargest(5, "eigenvector_mean")[["id", "eigenvector_mean", "age", "sex"]])
    
    # Coefficient plot
    fig, ax = plt.subplots(figsize=(10, 6))
    y = np.arange(len(summary))
    ax.barh(y, summary["mean"], color="#3498db", alpha=0.7)
    ax.errorbar(summary["mean"], y, xerr=[summary["mean"]-summary["2.5%"], summary["97.5%"]-summary["mean"]],
                fmt='none', ecolor='black', capsize=5, linewidth=2)
    ax.axvline(0, color='red', linestyle='--', linewidth=1)
    ax.set_yticks(y); ax.set_yticklabels(summary.index)
    ax.set_xlabel("Coefficient Value", fontsize=12)
    ax.set_title("Eigenvector Centrality ~ Age + Sex\n(Posterior Mean and 95% CI)", fontsize=14, fontweight="bold")
    ax.grid(axis='x', alpha=0.3)
    plt.tight_layout()
    plt.savefig(OUTDIR / "regression_coefficients_plot.png", dpi=200)
    plt.close()
    
    # Ridgeline plot
    node_order = node_df.sort_values("eigenvector_mean", ascending=False)["id"].tolist()
    g_max = eigen.max() * 1.1
    fig, axes = plt.subplots(len(nodes), 1, figsize=(10, len(nodes)*0.5), sharex=True)
    for idx, nid in enumerate(node_order):
        ax = axes[idx] if len(nodes) > 1 else axes
        samples = eigen[:, nodes.index(nid)]
        if samples.std() > 0:
            kde = gaussian_kde(samples, bw_method=0.3)
            x = np.linspace(0, g_max, 200)
            dens = kde(x) / kde(x).max()
        else:
            x, dens = np.array([samples.mean()]), np.array([1.0])
        ax.fill_between(x, 0, dens, color="#5A9", alpha=0.7, edgecolor="black", linewidth=0.5)
        ax.set_ylim(0, 1.2); ax.set_xlim(0, g_max); ax.set_yticks([])
        for spine in ['left', 'top', 'right']: ax.spines[spine].set_visible(False)
        ax.text(g_max*1.02, 0.5, str(nid), va='center', ha='left', fontsize=9, fontweight='bold')
        if idx < len(nodes)-1: ax.spines['bottom'].set_visible(False); ax.set_xticks([])
    axes[-1].set_xlabel("Eigenvector centrality", fontsize=11)
    plt.subplots_adjust(hspace=-0.5)
    plt.savefig(OUTDIR / "eigenvector_ridgeline.png", dpi=200, bbox_inches='tight')
    plt.close()
    
    # Network viz
    G = nx.Graph()
    lam_mean = lam_samples.mean(axis=0)
    for idx, row in dyads.iterrows():
        G.add_edge(row["node1"], row["node2"], weight=lam_mean[idx])
    for nid in nodes:
        if nid not in G: G.add_node(nid)
    pos = nx.spring_layout(G, weight="weight", seed=42, k=1.5, iterations=50)
    
    fig, ax = plt.subplots(figsize=(10, 10))
    ew = np.array([G[u][v]["weight"] for u, v in G.edges()])
    nx.draw_networkx_edges(G, pos, width=ew/ew.max()*5, alpha=0.6, edge_color=ew/ew.max(), 
                          edge_cmap=plt.cm.Greys, ax=ax)
    nx.draw_networkx_nodes(G, pos, node_color="#5A9", node_size=800, alpha=0.9, 
                          edgecolors="black", linewidths=2, ax=ax)
    nx.draw_networkx_labels(G, pos, {n: str(n) for n in G.nodes()}, font_size=10, 
                           font_weight="bold", font_color="white", ax=ax)
    ax.axis("off")
    ax.set_title("SRKW Social Network", fontsize=16, fontweight="bold", pad=20)
    plt.tight_layout()
    plt.savefig(OUTDIR / "network_visualization.png", dpi=300, bbox_inches="tight")
    plt.close()
    
    print(f"\n✓ Results saved to: {OUTDIR}")

def main():
    print("="*60)
    print("SRKW BISoN Pipeline - Eigenvector Centrality Analysis")
    print("="*60)
    
    attrs, contacts = load_data()
    dyads = aggregate_dyads(contacts)
    print(f"Aggregated to {len(dyads)} dyads")
    
    nodes = sorted(attrs["id"].unique())
    lam_samples, id2idx = fit_model(dyads, nodes)
    eigen_samples = propagate(lam_samples, dyads, id2idx)
    coefs_df = run_regression(eigen_samples, attrs, nodes)
    save_and_plot(eigen_samples, coefs_df, attrs, nodes, dyads, lam_samples, id2idx)
    
    print("\n" + "="*60)
    print("ANALYSIS COMPLETE!")
    print("="*60)

if __name__ == "__main__":
    main()

WARNING (pytensor.configdefaults): g++ not available, if using conda: `conda install gxx`
WARNING (pytensor.configdefaults): g++ not detected!  PyTensor will be unable to compile C-implementations and will default to Python. Performance may be severely degraded. To remove this warning, set PyTensor flags cxx to an empty string.


SRKW BISoN Pipeline - Eigenvector Centrality Analysis
Loaded 22 nodes, 89 contacts
Aggregated to 89 dyads

Fitting Bayesian model: 1000 draws, 800 tune, 2 chains


Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (2 chains in 2 jobs)
NUTS: [alpha_0, sigma_u, u, alpha_nb]


Output()

Sampling 2 chains for 800 tune and 1_000 draw iterations (1_600 + 2_000 draws total) took 684 seconds.
We recommend running at least 4 chains for robust computation of convergence diagnostics
The effective sample size per chain is smaller than 100 for some parameters.  A higher number is needed for reliable rhat and ess computation. See https://arxiv.org/abs/1903.08008 for details


Sampling complete!

Model summary:
           mean     sd  hdi_3%  hdi_97%  mcse_mean  mcse_sd  ess_bulk  \
alpha_0   2.113  0.197   1.750    2.481      0.007    0.006     870.0   
sigma_u   0.308  0.134   0.049    0.538      0.009    0.005     171.0   
alpha_nb  1.084  0.182   0.763    1.438      0.005    0.005    1133.0   

          ess_tail  r_hat  
alpha_0     1001.0   1.00  
sigma_u      107.0   1.01  
alpha_nb     938.0   1.00  

Propagating 200 draws to eigenvector centrality...
Complete! Shape: (200, 22)

Running regressions...

REGRESSION RESULTS: Eigenvector Centrality ~ Age + Sex
               mean      2.5%     97.5%
Intercept  0.295485  0.274390  0.313347
age       -0.006582 -0.007664 -0.005587
sex_M      0.000000  0.000000  0.000000

Top 5 nodes:
     id  eigenvector_mean  age sex
20  J53          0.370882    4   0
16  J46          0.349469   10   0
12  J41          0.290216   14   0
11  J40          0.278871   15   0
6   J35          0.275879   21   0

✓ Results saved 